# [ICM Development](@id Development)

## Definition of the model

### Mechanics

The cells are spheroids that behave under the following equations:

$$\lambda_s \mathbf{v}_i  + \lambda_r \sum_j (\mathbf{v}_i - \mathbf{v}_j) = \sum_j \mathbf{F}_{ij}$$

where the force is

$$F_{ij}=
\begin{cases}
F_{att/rep}(\frac{r_{ij}}{d_{ij}}-1)(\frac{\mu r_{ij}}{d_{ij}}-1)\frac{(x_i-x_j)}{d_{ij}}\hspace{1cm}if\;d_{ij}<\mu r_{ij}\\
0\hspace{5cm}otherwise
\end{cases}$$

where $d_{ij}$ is the Euclidean distance and $r_{ij}$ is the sum of both radius.

### Biochemical interaction

The model considers 3 cellular states: A,B and C. Transitions are unidirectional from A to B and then B to C. The transition rate for a given cell to transition is given by:
$$\dot{\phi}_A = \frac{p}{1+\phi_A/K}$$
$$\dot{\phi}_B = \frac{q}{1+\phi_A/K}$$

where $\phi_A$ is the fraction of A-type cells in the system and $K$ is the feedback paramteter.


### Growth

The cells present division. The rules for the division in this model are. Random election of a division direction over the unit sphere. The daughter cells divide equally in mass and volume and are positioned in oposite directions around the division axis centered at the parent cell. A new division time is assigned to each aghter cell from a uniform distribution $\text{Uniform}(\tau_{div}(1-\sigma_{div}),\tau_{div}(1+\sigma_{div}))$.

## Creation of the Agent

In [3]:
using CellBasedModels
using Random
using Distributions
using CSV
using Pkg
using WriteVTK
using DataFrames, Distances, Clustering
using GeometryBasics
using Colors
using StaticArrays
using Statistics
using Dates

### Define the agent

First, we have to create an instance of an agent with all the propoerties of the agents.First, we have to create an instance of an agent with all the propoerties of the agents.

In [ ]:
model = ABM(3,
   
# Global parameters
    model = Dict(

        :lambda=>Float64,
        :mu=>Float64,
        :F_att=>Array{Float64},
        :F_rep=>Array{Float64},
        
        :kpON_=>Array{Float64},     # Protrusion rate/"probability"
        :kpOFF_=>Array{Float64},    # Protrusion duration rate/"probability"
        :fRange_P=>Float64,         # Protrusion force interaction range factor
        :fRange_P_min=>Float64,     # Protrusion force minimum range factor for establishing protrusion bond
        :fRange_P_break=>Float64,   # Protrusion force break range factor
        :fPI=>Float64,              # Contractile Protrusion force magnitude
        :p=>Float64,                # Transition rate from A to B
        :q=>Float64,                # Transition rate from B to C
        :K=>Array{Float64},         # Transition rate feedback factor constant
        :fPI_=>Array{Float64},      #  Contractile Protrusion force magnitude

    # Division constants:
        :τDiv_=>Array{Float64},
        :τDiv_pre_=>Array{Float64},
        :σDiv_=>Array{Float64},

    # velocity dissipation:
        :vel_diss=>Float64,
        
    # Physical constants
        :fRange=>Float64,
        :ri=>Float64, 

    # Division constants
        :τDiv=>Float64, 
        :σDiv=>Float64,  
    ),

    
# Local float parameters
    agent = Dict(

        :r=>Float64,
        :vx=>Float64,
        :vy=>Float64,
        :vz=>Float64,
        :fx=>Float64,
        :fy=>Float64,
        :fz=>Float64,
        
        #############################################################################################################################
        :fpx=>Float64,
        :fpy=>Float64,
        :fpz=>Float64,
        
        :nNeighs=>Int64,    # Number of neighbours that the agent has for velocity term in forces
        :nNeighs_A=>Int64,  # Number of neighbours that the agent has in state A (cellFate = 1)
        :nNeighs_B=>Int64,  # Number of neighbours that the agent has in state B (cellFate = 2)
        :nNeighs_C=>Int64,  # Number of neighbours that the agent has in state C (cellFate = 3)
        :nNeighs_P=>Int64,  # Number of neighbours that the agent has for Protrusion term in forces
        :nSumVx=>Float64,   # Sum of neighbours' vx
        :nSumVy=>Float64,   # Sum of neighbours' vy
        :nSumVz=>Float64,   # Sum of neighbours' vz

        :kp=>Float64,       # Cell protrusion random number
        :τp=>Float64,       # Protrusion time constant
        :tij=>Float64,      # Protrusion time 
        :tij_c=>Float64,    # Protrusion time counter

    #For controlling division:
        :cell_dividing=>Int64,
        :cell_div_relax_steps=>Int64,

        :salt_and_pepper_switch=>Int64,

        :N_start_sim_switch=>Int64,

        :N_diff_min_switch=>Int64,

        :prolif_active_switch=>Int64,

        :N_active=>Int64, # com.cellFate components out of Nmax that are actually there!
        :N_state1=>Int64, # Number of com.cellFate components that are = 1 (state A)!

        :prolif_active=>Int64, # Tell if cell is chosen randomly for proliferation
        
        :tDivision=>Float64, #Variable storing the time of division of the cell
        :cellFate=>Int64 #Identity of the cell (1 A, 2 B, 3 C)
    ),


    
###Mechnical & Chemical dynamics
    agentODE = quote
        
        #############################################################################################################################
        #Mechanics
        fx = 0.0; fy = 0.0; fz = 0.0
        fmax = 1000.0  # corresponds to a distance of R/8 between the cells
        switch_fmax_x = 0
        switch_fmax_y = 0
        switch_fmax_z = 0
        @loopOverNeighbors it2 begin
            dij = sqrt((x-x[it2])^2+(y-y[it2])^2+(z-z[it2])^2)
            rij = r+r[it2]
            
            if dij < mu*rij && dij > 0
                
                if dij < rij

                    fx += ( F_rep[cellFate[i1_],cellFate[it2]] * (rij/dij-1)*(mu*rij/dij-1) ) * (x-x[it2])/dij 
                    fy += ( F_rep[cellFate[i1_],cellFate[it2]] * (rij/dij-1)*(mu*rij/dij-1) ) * (y-y[it2])/dij
                    fz += ( F_rep[cellFate[i1_],cellFate[it2]] * (rij/dij-1)*(mu*rij/dij-1) ) * (z-z[it2])/dij

                    # softer force:
                    if fx > fmax
                        fx = fmax
                        if switch_fmax_x == 0
                            println(" **********   fx REACHED fmax!!!   **********")
                            switch_fmax_x = 1
                        end
                    end
                    if fy > fmax
                        fy = fmax
                        if switch_fmax_y == 0
                            println(" **********   fy REACHED fmax!!!   **********")
                            switch_fmax_y = 1
                        end
                    end
                    if fz > fmax
                        fz = fmax
                        if switch_fmax_z == 0
                            println(" **********   fz REACHED fmax!!!   **********")
                            switch_fmax_z = 1
                        end
                    end
                
                else

                # Original force:
                    #1#:
                    fx += F_att[cellFate[i1_],cellFate[it2]]*(rij/dij-1)*(mu*rij/dij-1)*(x-x[it2])/dij
                    fy += F_att[cellFate[i1_],cellFate[it2]]*(rij/dij-1)*(mu*rij/dij-1)*(y-y[it2])/dij
                    fz += F_att[cellFate[i1_],cellFate[it2]]*(rij/dij-1)*(mu*rij/dij-1)*(z-z[it2])/dij

                # softer force:
                    if fx > fmax
                        fx = fmax
                        if switch_fmax_x == 0
                            println(" **********   fx REACHED fmax!!!   **********")
                            switch_fmax_x = 1
                        end
                    end
                    if fy > fmax
                        fy = fmax
                        if switch_fmax_y == 0
                            println(" **********   fy REACHED fmax!!!   **********")
                            switch_fmax_y = 1
                        end
                    end
                    if fz > fmax
                        fz = fmax
                        if switch_fmax_z == 0
                            println(" **********   fz REACHED fmax!!!   **********")
                            switch_fmax_z = 1
                        end
                    end

                end
                
            end

            
        end
        
        if nNeighs < N_relV_min

            dt(x) = fx/lambda + fpx/lambda
            dt(y) = fy/lambda + fpy/lambda
            dt(z) = fz/lambda + fpz/lambda

            vx = dt(x)
            vy = dt(y)
            vz = dt(z)
           
        else
       
            dt(x) = fx/nNeighs/lambda + nSumVx/nNeighs + fpx/nNeighs/lambda
            dt(y) = fy/nNeighs/lambda + nSumVy/nNeighs + fpy/nNeighs/lambda
            dt(z) = fz/nNeighs/lambda + nSumVz/nNeighs + fpz/nNeighs/lambda 

            vx = dt(x)
            vy = dt(y)
            vz = dt(z)
            
        end
    

        compile=false
        #############################################################################################################################

    end,

    
    agentRule=quote

    # If we want to list the nNeighs:
        nNeighs_list = zeros(Int64, 0)
        
        nNeighs_P_list = zeros(Int64, 0)

            
        fpx = 0.; fpy = 0.; fpz = 0.
    # Count neighbours for each cell & Add neighbour velocities for primary cell velocity calculation
        nNeighs_new = 0 #Set it to zero before starting the computation
        nNeighs_A_new = 0 #Set it to zero before starting the computation
        nNeighs_B_new = 0 #Set it to zero before starting the computation
        nNeighs_C_new = 0 #Set it to zero before starting the computation
        nNeighs_P_new = 0 #Set it to zero before starting the computation
        nSumVx_new = 0. #Set it to zero before starting the computation
        nSumVy_new = 0. #Set it to zero before starting the computation
        nSumVz_new = 0. #Set it to zero before starting the computation
        nSumV_max = 100.0 #max vel force
        switch_nSumVx_max = 0
        switch_nSumVy_max = 0
        switch_nSumVz_max = 0
        halt_division = 0

        
        
    # FOR INITIALISING SALT & PEPPER DISTRIBUTION: 
        if N == N_start_sim && salt_and_pepper == 1 && salt_and_pepper_switch == 0

            if i1_ in selected_indices
                cellFate = 2
            end
            salt_and_pepper_switch = 1
        end
        
        
        @loopOverNeighbors it2 begin
           
            d = CBMMetrics.euclidean(x,x[it2],y,y[it2],z,z[it2]) #Using euclidean matric provided in package
            ndist = (r + r[it2])*fRange
          
        # For Cell Velocity update:
            if d < ndist

                nNeighs_new += 1 #Add 1 to neighbors of cell

                if cellFate[it2] == 1
                    nNeighs_A_new += 1 #Add 1 to A-neighbors of cell
                elseif cellFate[it2] == 2
                    nNeighs_B_new += 1 #Add 1 to B-neighbors of cell
                elseif cellFate[it2] == 3
                    nNeighs_C_new += 1 #Add 1 to C-neighbors of cell
                end

                if cell_dividing == 0 && cell_dividing[it2] == 0
                
                    nSumVx_new += vx[it2]*vel_diss
                    nSumVy_new += vy[it2]*vel_diss
                    nSumVz_new += vz[it2]*vel_diss
                        
                elseif cell_dividing > 0
                    
                    cell_dividing += 1
                    if cell_dividing > cell_div_relax_steps
                        cell_dividing = 0
                    end
                    
                end
                

            # If we want to list the nNeighs: 
                push!(nNeighs_list, it2)

            end

        # For Protrusion Pair Interaction:
            ndist_P = r*fRange_P   # this is assuming all cells have the same radius! This gives FCC 3rd NNs! (Just before cells are allowed to pull other cells exactly behind their 1st NN (d=4r))
          
            if d < ndist_P && d > fRange_P_min*r
                nNeighs_P_new += 1

                push!(nNeighs_P_list, it2)

            end
        
        end

        

        
    # For Protrusion Force Activation:

        if nNeighs_P_new != 0
            random_neigh = Int64(ceil(CBMDistributions.uniform(0,1) * length(nNeighs_P_list)))
            d = CBMMetrics.euclidean(x,x[random_neigh],y,y[random_neigh],z,z[random_neigh])
        end
        
        if nNeighs_P_new != 0 && kp < (kpON_[cellFate[i1_],cellFate[random_neigh]]*dt) && PI[i1_] == 0 && d < ndist_P && d >= fRange_P_min*r && N >= N_prot_min && prot_ON == 1

            tij_new = τp - log(CBMDistributions.uniform(0,1)) / kpOFF_[cellFate[i1_],cellFate[random_neigh]]

        # ALLOW ONE PROTRUSION PER CELL, BUT MORE THAN ONE BOND:
            tij = tij_new
            tij_c_new = 0.0
            tij_c = tij_c_new
        
        # ALLOW ONE PROTRUSION PER CELL, BUT MORE THAN ONE BOND:
            PI[i1_] = random_neigh
            append!(PI_list[random_neigh], [i1_])
           
        else
            kp_new = CBMDistributions.uniform(0,1)
            kp = kp_new
        end


    # Finally, update nNeighs and Neighbour Velocity Sums for cell:
        
        nNeighs = nNeighs_new
        nNeighs_A = nNeighs_A_new
        nNeighs_B = nNeighs_B_new
        nNeighs_C = nNeighs_C_new
        
        nSumVx = nSumVx_new
        nSumVy = nSumVy_new
        nSumVz = nSumVz_new

        nNeighs_P = nNeighs_P_new

        
    # Protrusion Force Profile:
       
        if (PI[i1_] != 0) && (N >= N_prot_min) && prot_ON == 1
           
            cell_partner = abs(PI[i1_])
           
            d_partner = CBMMetrics.euclidean(x,x[cell_partner],y,y[cell_partner],z,z[cell_partner])
            ndist_P = r * fRange_P # Assuming FCC structure with all particles having same radius! 
            
            if tij_c < tij && d_partner < ndist_P && d_partner >= fRange_P_break*r
            
                tij_c += dt

                fpx -= fPI_[cellFate[i1_],cellFate[cell_partner]] * (x-x[cell_partner])/d_partner
                fpy -= fPI_[cellFate[i1_],cellFate[cell_partner]] * (y-y[cell_partner])/d_partner
                fpz -= fPI_[cellFate[i1_],cellFate[cell_partner]] * (z-z[cell_partner])/d_partner
              
            elseif tij_c >= tij || d_partner >= ndist_P || d_partner < fRange_P_break*r
            
                PI[i1_] = 0
                tij_c_new = 0.0
                tij_c = tij_c_new
                kp_new = CBMDistributions.uniform(0,1)
                kp = kp_new

                idx = findfirst(==(i1_), PI_list[cell_partner])
                if idx !== nothing
                    deleteat!(PI_list[cell_partner], idx)
                end
                
            end
        end

        
    # Compute force due to cell being bonded to other cells that extended prots to it:
        
        if (N >= N_prot_min) && prot_ON == 1
            for cell_partnered_to in PI_list[i1_]
                    
                d = CBMMetrics.euclidean(x,x[cell_partnered_to],y,y[cell_partnered_to],z,z[cell_partnered_to])
                ndist_P = r * fRange_P # Assuming FCC structure with all particles having same radius!
        
                if tij_c[cell_partnered_to] < tij[cell_partnered_to] && d != 0 && d < ndist_P && d >= fRange_P_break*r
                    
                    fpx -= fPI_[cellFate[cell_partnered_to],cellFate[i1_]] * (x-x[cell_partnered_to])/d
                    fpy -= fPI_[cellFate[cell_partnered_to],cellFate[i1_]] * (y-y[cell_partnered_to])/d
                    fpz -= fPI_[cellFate[cell_partnered_to],cellFate[i1_]] * (z-z[cell_partnered_to])/d

                elseif tij_c[cell_partnered_to] >= tij[cell_partnered_to] || d >= ndist_P || d < fRange_P_break*r

                    PI[cell_partnered_to] = 0
                    tij_c_new = 0.0
                    tij_c[cell_partnered_to] = tij_c_new
                    kp_new = CBMDistributions.uniform(0,1)
                    kp[cell_partnered_to] = kp_new

                    idx = findfirst(==(cell_partnered_to), PI_list[i1_])
                    if idx !== nothing
                        deleteat!(PI_list[i1_], idx)
                    end
                       
                end
                   
            end
        end
        #############################################################################################################################
       


        
    #Differentiation
        if N >= N_diff_min && diff_ON == 1

        # total active (non-zero) cells (Since I have Nmax slots, some will be 0 if N < Nmax !)
            N_active = count(!=(0), com.cellFate)

        # number of cells in state 1
            N_state1 = count(==(1), com.cellFate)
   

            if N >= N_start_sim && N_start_sim_switch == 0 && N_diff_min_switch < 2

               N_diff_min_switch += 1

            elseif  N >= N_start_sim && N_start_sim_switch == 1 && N_diff_min_switch < 2 && N == (N_start_sim + 50)

               N_diff_min_switch += 1
                
            end
            
            rand_number = CBMDistributions.uniform(0,1)
        
        # MEAN FIELD APPROACH:
            if nNeighs > 0 && cellFate == 1 && rand_number < ( p*dt / (1.0 + K[cellFate] * N_state1 / N_active) )
                cellFate = 2
            elseif nNeighs > 0 && cellFate == 2 && rand_number < ( q*dt / (1.0 + K[cellFate] * N_state1 / N_active) )
                cellFate = 3
            end
            
        end


        
    #Proliferation
        
        if N >= N_start_sim && N_start_sim_switch == 0
            τDiv_value = τDiv_[cellFate]
            if N_start_sim_switch == 0
                tDivision = t + CBMDistributions.uniform(τDiv_value*(1-σDiv_[cellFate]),τDiv_value*(1+σDiv_[cellFate]))
            end
            N_start_sim_switch = 1
        end


        if N_start_sim_switch == 0
            τDiv_value = τDiv_pre_[cellFate]
        else
            τDiv_value = τDiv_[cellFate]
        end



        
        
        if t > tDivision && N < Nmax && prolif_active == 1
                
            if halt_division == 0
            
                if PI[i1_] != 0
                    cell_partner = abs(PI[i1_])
                    PI[i1_] = 0
                    PI[cell_partner] = 0
                    tij_c = 0
                    tij_c[cell_partner] = 0
                    kp[cell_partner] = CBMDistributions.uniform(0,1)
                end
    
            #Choose random direction in unit sphere
                xₐ = CBMDistributions.normal(0,1); yₐ = CBMDistributions.normal(0,1); zₐ = CBMDistributions.normal(0,1)
                Tₐ = sqrt(xₐ^2+yₐ^2+zₐ^2)
                xₐ /= Tₐ; yₐ /= Tₐ; zₐ /= Tₐ    
    
            #Chose a random distribution of the material
                rnew = r
                rsep = 0.3*r 
                    
                @addAgent(          # add new agent
                    x = x+rsep*xₐ,
                    y = y+rsep*yₐ,
                    z = z+rsep*zₐ,
                
                    vx = vx/2.,
                    vy = vy/2.,
                    vz = vz/2.,
    
                    r = rnew,
                    tDivision = t + CBMDistributions.uniform(τDiv_value*(1-σDiv_[cellFate]),τDiv_value*(1+σDiv_[cellFate])),
                    kp = CBMDistributions.uniform(0,1),
                    tij = 0.0,
                    cell_dividing = 1,
                )
                @addAgent(          # add new agent
                    x = x-rsep*xₐ,
                    y = y-rsep*yₐ,
                    z = z-rsep*zₐ,
                   
                    vx = vx/2.,
                    vy = vy/2.,
                    vz = vz/2.,
                    
                    r = rnew,
                    tDivision = t + CBMDistributions.uniform(τDiv_value*(1-σDiv_[cellFate]),τDiv_value*(1+σDiv_[cellFate])),
                    kp = CBMDistributions.uniform(0,1),
                    tij = 0.0,
                    cell_dividing = 1,
                )
                @removeAgent()      # remove agent that divided
            
            else

                halt_division = 0
                
            end

        end

    end,

    agentAlg=CBMIntegrators.Heun()
);

## Community construction and initialisation

Once with the model created, we have to construct an initial Community of agents to evolve.

### Parameters

The model from the original version has some parameters defined. We create a dictionary with all the parameters from the model assigned.

In [ ]:
parameters = Dict([

    :fRange => 1.0,
    :ri => 1.0,
    :fRange_P => 2.0,
    :fRange_P_min => 2.0,
    :fRange_P_break => 1.0,
    :lambda => 1E+0,   
    :kpON_ => [0.0 0.0 0.0; 1.0 1.0 1.0; 5.0 5.0 5.0],
    :kpOFF_ => [200.0 200.0 200.0; 2.0 0.1 2.0; 2.0 2.0 2.0],
    :τp => 0.0,
    :p => 0.009333,
    :q => 0.004667,
    :K => [5.0 5.0 0.0],
    :vel_diss => 0.9, # velocity/momentum dissipation for relative velocity
    :mu => 2.0, #this must be greater than 1 for the model to make sense!
    :τDiv_pre_ => [5 5 5],
    :τDiv_ => [300 300 750],
    :σDiv_ => [0.5 0.5 0.5],
    :F_att => [6.0 6.0 2.4; 6.0 6.0 1.6; 2.4 1.6 4.8],
    :F_rep => [3.0 3.0 3.0; 3.0 3.0 3.0; 3.0 3.0 5.1],
    :fPI_ => [5 2 5; 2 5 2; 5 7 7], # protrusion forces
        
]);

### Initialise the community

The model starts from just one agent. Create the community and assign all the parameters to the Community object.

In [6]:
function initializeEmbryo(parameters;dt,N)

    com = Community(
                model,
                N=N,
                dt=dt,
                )

    #Global parameters
    for (par,val) in pairs(parameters)
        com[par] = val
    end
    
    #Initialise locals
    com.r = parameters[:ri]
    com.F_att = parameters[:F_att]
    com.cellFate = 1 #Start neutral (A) fate 
    com.cell_dividing = 0
    com.cell_div_relax_steps = cell_div_relax_steps
    com.salt_and_pepper_switch = 0
    com.N_start_sim_switch = 0
    com.N_diff_min_switch = 0
    com.prolif_active_switch = 0
    com.N_state1 = 1 # should be 1
    com.N_active = 1 # should be 1
    com.prolif_active = 1
   
    #Initialise variables
    #############################################################################################################################
    com.nNeighs = 0 #Start with the nNeighs = 0
    com.nNeighs_A = 0 #Start with the nNeighs_A = 0
    com.nNeighs_B = 0 #Start with the nNeighs_B = 0
    com.nNeighs_C = 0 #Start with the nNeighs_C = 0
    com.nNeighs_P = 0 #Start with the nNeighs_P = 0
    com.nSumVx = 0. #Start with the nSumVx = 0.
    com.nSumVy = 0. #Start with the nSumVy = 0.
    com.nSumVz = 0. #Start with the nSumVz = 0.
    com.tij = 0. #Start with tij = 0.
    com.tij_c = 0. #Start with tij = 0.
    com.kp = CBMDistributions.uniform(0,1) #Start with random vlaues of kp drawn from uniform dist.
    #############################################################################################################################
    com.x = 0.
    com.y =  0.
    com.z =  0.
    com.vx = 0.
    com.vy = 0.
    com.vz = 0.
    com.tDivision = 1 #rand(Uniform(com.τDiv-com.σDiv,com.τDiv+com.σDiv))

    return com

end;

## Creating a custom evolve step

In [7]:
function customEvolve!(com,steps,saveEach)
    loadToPlatform!(com,preallocateAgents = Nmax-N_ini+1) #loadToPlatform!(com,preallocateAgents = 100)
    switch_0 = 0
    switch_1 = 0
    switch_2 = 0
    for i in 1:steps

        if i % 1000 == 0
            println("      ***** i = ", i, " / ", steps, " and com = " , com.N, " agents *****    \n")
            switch_1 = 0
            switch_2 = 0            
        end

        

        agentStepDE!(com)
        agentStepRule!(com)
        update!(com)
        computeNeighbors!(com)
        
        if i % saveEach == 0
            saveRAM!(com)
        end
        
        #Stop by time
        if (switch_0 == 0) && all(com.N .>= Nmax)  # if all(com.N .> 60)
            println("\n\n\n --------- REACHED THRESHHOLD! com.N = ",com.N, " REACHED THRESHHOLD! ---------")
            switch_0 = 1
        end
        
    end
    bringFromPlatform!(com)
    
end;

# Setup simulation

We check how the agents starts to divide and choose a fate at late stages of the simulation.

In [8]:
random_seed = 74375
Random.seed!(random_seed)

rng = MersenneTwister(random_seed)  # Create an explicit RNG instance with the seed


dt = 0.02 #0.001 #0.005 #0.0005  ##0.0002  ###0.005   #small early stage tests: 0.0005

# Number of cells at start of simulation
N_ini = 1 # minimum value of 1!

# Maximum number of cells
Nmax = 15000

# Number of cells after which real simulation starts:
N_start_sim = 300

# Active cell-cell protrusions
prot_ON = 0

# When protrusions kick in
N_prot_min = N_start_sim

# Cell differentiation
diff_ON = 1
N_diff_min = N_start_sim

# Relative velocities
N_relV_min = 1  #When relative friction kicks in.
cell_div_relax_steps = 150




# TO CREATE SALT & PEPPER DISTRIBUTION OF CELL FATES (ONLY WORKS WHEN STARTING WITH 100% A-TYPE AGGREGATES!):

salt_and_pepper = 0
S_and_P_B_proportion = 0.0


if salt_and_pepper == 1
    
    # How many of those to convert?
    num_to_convert = round(Int, N_start_sim * S_and_P_B_proportion)
    
    # Select which ones to convert randomly
    selected_indices = randperm(rng, N_start_sim)[1:num_to_convert]  # Random indices from type 1
    
end


PI = zeros(Int64, Nmax)

# If we want to list the nNeighs:
nNeighs_list = zeros(Int64, 0)
nNeighs_P_list = zeros(Int64, 0)


# If we want to know which cells are bonded by prots to chosen cell:
PI_list = [Int64[] for _ in 1:Nmax]


steps = round(Int64,1300/dt)
saveEach = round(Int64,1/dt)


com = initializeEmbryo(parameters,dt=dt,N=N_ini);
customEvolve!(com,steps,saveEach)

println(" ")
println("Finished simulation. ",com)


      ***** i = 1000 / 65000 and com = 28 agents *****    

      ***** i = 2000 / 65000 and com = 301 agents *****    

      ***** i = 3000 / 65000 and com = 301 agents *****    

      ***** i = 4000 / 65000 and com = 301 agents *****    

      ***** i = 5000 / 65000 and com = 301 agents *****    

      ***** i = 6000 / 65000 and com = 301 agents *****    

      ***** i = 7000 / 65000 and com = 301 agents *****    

      ***** i = 8000 / 65000 and com = 301 agents *****    

      ***** i = 9000 / 65000 and com = 301 agents *****    

      ***** i = 10000 / 65000 and com = 325 agents *****    

      ***** i = 11000 / 65000 and com = 350 agents *****    

      ***** i = 12000 / 65000 and com = 372 agents *****    

      ***** i = 13000 / 65000 and com = 382 agents *****    

      ***** i = 14000 / 65000 and com = 394 agents *****    

      ***** i = 15000 / 65000 and com = 416 agents *****    

      ***** i = 16000 / 65000 and com = 431 agents *****    

      ***** i = 17

### Make directory for saving output files 



In [9]:
dir_name = "results"
mkpath(dir_name)

"results"

# Save cell positions and fates with time



In [10]:
d = getParameter(com,[:x,:y,:z,:cellFate]);
df = DataFrame(d);

In [11]:
using DataFrames

times_expanded = Int[]
x_expanded = Float64[]
y_expanded = Float64[]
z_expanded = Float64[]
fate_expanded = Int[]

for (i, row) in enumerate(eachrow(df))  # i is timestep index
    n_cells = length(row.x)
    
    append!(times_expanded, fill(i, n_cells))  # use i as time
    
    append!(x_expanded, row.x)
    append!(y_expanded, row.y)
    append!(z_expanded, row.z)
    append!(fate_expanded, row.cellFate)
end

flat_df = DataFrame(t = times_expanded,
                    x = x_expanded,
                    y = y_expanded,
                    z = z_expanded,
                    cellFate = fate_expanded)

# Now save as CSV
using CSV
CSV.write("$dir_name/cells.csv", flat_df)

"results/cells.csv"